In [1]:
import os
from dotenv import load_dotenv
from langsmith import Client
opanai_key = os.getenv("OPENAI_PI_KEY")
langsmith_key = os.getenv("LANGSMITH_API_KEY")
langsmith_tracing = True
load_dotenv(override=True)

True

In [2]:
client = Client()
dataset_name = "Simple Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id= dataset.id,
    examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }        
    ]
)

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

## Define Metrics (LLM as a Judge) ##

In [3]:
import openai
from langsmith import wrappers

openai_client =wrappers.wrap_openai(openai.OpenAI())
eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict)-> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature = 0,
        messages = [
            {"role":"system", "content":eval_instructions},
            {"role": "user", "content": user_content}
        ]
    ).choices[0].message.content
    
    return response == "CORRECT"
    

In [4]:
# Concisions : Checks whether the actual output is less than 2x the length of expected result.

def concision(outputs: dict, reference_outputs: dict)-> bool:
    return int(len(outputs["response"])) < 2 * len(reference_outputs["answer"])

In [5]:
## Running the Evaluation ##
default_instructions = "Respond to the users question in a short, concise manner(one short sentence)."
def my_app(question: str, instructions: str = default_instructions)-> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role":"system", "content": instructions},
            {"role":"user", "content":question}
        ]
    ).choices[0].message.content
    return response

In [6]:
# calling my_app for every datapoints
def ls_target(inputs: str)-> dict:
    return {"response": my_app(inputs["question"])}

In [7]:
## Run for evaluation
experimental_results= client.evaluate(
    ls_target, # AI System
    data = dataset_name,
    evaluators = [correctness, concision],
    experiment_prefix= "openai-4o-mini"
)

experimental_results

View the evaluation results for experiment: 'openai-4o-mini-25799e44' at:
https://smith.langchain.com/o/ded64c70-97b2-4828-9487-f10ed0ed0c43/datasets/224c90e8-6402-44c9-afa8-5487c04ffac4/compare?selectedSessions=bd1a6562-2932-4605-8bd1-9667dc4e04f7




c:\work\AgenticAI\llm-evaluation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
5it [00:13,  2.73s/it]


<ExperimentResults openai-4o-mini-25799e44>

## Evaluation For RAG ##

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of Urls to load documents from

urls = [
    "https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/a.pdf",
    "https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/b.pdf",
    "https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/c.pdf",
]

#load documents from Urls
docs = [PyPDFLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

#Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

#split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

#Add the document chunks to the "vector store" using the OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents = doc_splits,
    embedding = OpenAIEmbeddings()
)

#with using langchain we can turn vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

C:\Users\AbhishekMungekar\AppData\Local\Temp\ipykernel_17932\1978945610.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
retriever.invoke("What is agents?")

[Document(id='f7262004-63bb-4ebf-b013-c545e3fd31d9', metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'LLM Powered Autonomous Agents', 'source': 'https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/a.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content="LLM Powered Autonomous AgentsDate: June 23, 2023  |  Author: Lilian Weng\nSource: Lil'Log (https://lilianweng.github.io/posts/2023-06-23-agent/)\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several\nproof-of-concepts  demos,  such  as  AutoGPT,  GPT-Engineer  and  BabyAGI,  serve  as  inspiring\nexamples. The potentiality of LLM extends beyond generating well-written copies, stories, essays\nand programs; it can be framed as a powerful general problem solver .\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent's brain, complemented\nby several key components:\nPlanning:\nSubgoal a

In [9]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("openai:gpt-4o-mini")
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001E559D5C9B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E559D5CC50>, root_client=<openai.OpenAI object at 0x000001E55A5B2150>, root_async_client=<openai.AsyncOpenAI object at 0x000001E559D5CC20>, model_name='gpt-4o-mini', model_kwargs={}, 

In [10]:
from langsmith import traceable

@traceable
def rag_bot(question: str)-> dict:
    ## Relevant context
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)
    
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use two sentences maximum and keep the answer concise.
                        Documents: {docs_string}"""
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ])
    return {"answer": ai_msg.content, "documents": docs}                        

In [11]:
rag_bot("What is agents")

{'answer': 'Agents refer to systems, often powered by large language models (LLMs), that can autonomously perform tasks by planning, reflecting, and utilizing memory. They break down complex tasks into manageable subgoals and use tools and external APIs to enhance their functionality.',
 'documents': [Document(id='aecde8e1-c62c-40b8-923a-1ae7e27d24b4', metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'LLM Powered Autonomous Agents', 'source': 'https://github.com/mungekarabhishek/llm-evaluation/raw/master/data/a.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content="LLM Powered Autonomous AgentsDate: June 23, 2023  |  Author: Lilian Weng\nSource: Lil'Log (https://lilianweng.github.io/posts/2023-06-23-agent/)\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several\nproof-of-concepts  demos,  such  as  AutoGPT,  GPT-Engineer  and  BabyAGI,  serve  as  inspiring\nexamples. The potentiality of 

### Dataset for RAG Evaluation ###

In [12]:
from langsmith import Client
client = Client()

examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {
            "answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."
        },
    },
    {
        "inputs": {
            "question": "What are the types of biases that can arise with few-shot prompting?"
        },
        "outputs": {
            "answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."
        },
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {
            "answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."
        },
    },
]


dataset_name = 'Evalation for RAG'
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = examples
)

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

In [13]:
from typing_extensions import Annotated, TypedDict

class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the socre"]
    correct: Annotated[str, ..., "True if the answer is correct, False Otherwise"]
    
## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_openai import ChatOpenAI

grader_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(CorrectnessGrade, method="json_schema", strict=True)

# evaluator

def correctness(inputs: dict, outputs: dict, reference_outputs: dict)->bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
        Question: {inputs['question']}
        Ground Truth Answer: {reference_outputs['answer']}
        Student Answer: {outputs['answer']}
        """
        
    grade = grader_llm.invoke([
            {"role": "system", "content": correctness_instructions},
            {"role": "user", "content": answers}
        ])
        
    return grade["correct"]    

    

## Relevance: Response vs Input  ##

The flow is similar to above, but simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [15]:
#Grade Output Schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]


# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

#Evaluator
def relevance(inputs: dict, outputs: dict)-> bool:
    """A simple evalutor for RAG answer helpfulness."""
    answer = f"Question: {inputs['question']}\n Student Answer: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade['relevant']

### Groundness : Response vs retrieved Docs ###
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [16]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoing for the score"]
    relevant: Annotated[str, ..., "True if the retrieved documents are relevant to the question, False otherwise"]
    
# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM

grounded_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

#Evaluator
def groundness(inputs: dict, outputs: dict)-> bool:
    """A simple evaluator for RAG answer groundness"""
    doc_string ="\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"Facts: {doc_string}\n Student Answer: {outputs['answer']}"
    grade = grounded_llm.invoke(
        [
            {"role": "system", "content": grounded_instructions},
            {"role": "user", "content": answer}
        ]
    )
    return grade["relevant"]

### Retrieval Relevance: Retrived docs vs input ###

In [17]:
# Grade Output Schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_releavance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relavance_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relavance(inputs: dict, outputs: dict)-> bool:
    """An evaluator for document relevance"""
    doc_string= "\n\n".join(doc.page_content for doc in outputs['documents'])
    answer = f"Facts: {doc_string}\n Question: {inputs['question']}"
    
    # Run Evaluator
    grade = retrieval_relavance_llm.invoke([
        {"role": "system", "content": retrieval_releavance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [21]:
def target(inputs: dict)-> dict:
    return rag_bot(inputs['question'])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundness, relevance, retrieval_relavance],
    experiment_prefix = "rag_doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"}
)

experiment_results.to_pandas()

View the evaluation results for experiment: 'rag_doc-relevance-85ff1068' at:
https://smith.langchain.com/o/ded64c70-97b2-4828-9487-f10ed0ed0c43/datasets/c5be0272-2155-4b47-ab5b-29e3ecf4de41/compare?selectedSessions=f81b036f-a18e-4a64-9b05-c5e03e6e8d9f




3it [00:35, 11.91s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundness,feedback.relevance,feedback.retrieval_relavance,execution_time,example_id,id
0,What are five types of adversarial attacks?,The five types of adversarial attacks mentione...,[page_content='Adversarial Attacks on LLMs\nDa...,None,Five types of adversarial attacks are (1) Toke...,False,True,True,True,3.538297,26319287-4219-430d-bbcf-4172c538af43,019eabb5-d22a-7fe1-9f53-85ea4592615a
1,What are the types of biases that can arise wi...,The document does not specify the types of bia...,[page_content='Sentiment:\nFew-shot prompting...,None,The biases that can arise with few-shot prompt...,False,False,False,False,1.862790,68817aa5-fda9-4b67-8f42-9fe67e3f2f7c,019eabb6-0627-7b30-8a60-2d71272bd6de
2,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection by integr...,"[page_content='structures, letting the LLM tra...",None,"ReAct integrates reasoning and acting, perform...",True,True,True,True,2.555019,eeb5bbeb-9ce5-4c77-ae7f-433419f963f6,019eabb6-2c34-72f0-81d4-0fde3ef0d5c8
